# 🚀 Notebook 06 — Advanced Gradient Boosting Model Training Suite

<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 20px; border-radius: 10px; color: white; margin: 10px 0;'>

**Advanced GBDT Model Training Engine** — Fits high-capacity Gradient Boosted Decision Tree architectures (LightGBM, CatBoost, XGBoost) with GPU acceleration and early stopping.

</div>

| Property | Value |
|:---|:---|
| 🏗️ **Target** | `unit_sales` (log-transformed $\log(1 + y)$ during model fitting) |
| 📥 **Input** | `feature_store.parquet` (74 engineered features) |
| ⚡ **Hardware** | Automatic Hardware Detection: NVIDIA CUDA GPU (XGBoost/CatBoost) \| CPU Multi-Threading 12 Cores (LightGBM) |
| 📤 **Model Artifacts** | Saved to `03_Models/advanced_models/*.joblib` |
| 🖼️ **Plots Directory** | Saved to `output/06_model_training/` |
| 🤖 **Algorithms** | LightGBM (Leaf-wise), CatBoost (Ordered Boosting), XGBoost (Histogram/CUDA) |

---

### 📑 Table of Contents

| # | Section | Technical Purpose |
|:---:|:---|:---| 
| 1 | Environment Setup | Bootstrap project paths and Python environment |
| 2 | System Setup & Hardware Check | Detect NVIDIA CUDA GPU, CPU cores, load dataset & split |
| 2.1 | Data Load & Split | Chronological out-of-time train/val split & $\log(1+y)$ target scaling |
| 2.2 | Metric Evaluation Engine | Standardized RMSLE, RMSE, MAE, MAPE, $R^2$, and runtime tracking |
| 3 | LightGBM Regressor | Leaf-wise GBDT training with OpenMP multi-threading & feature importances |
| 4 | CatBoost Regressor | Ordered GBDT training with native GPU acceleration & feature importances |
| 5 | XGBoost Regressor | Histogram CUDA GBDT training with early stopping & feature importances |
| 6 | Advanced Leaderboard | Comparative benchmark leaderboard GBDTs vs Baseline Random Forest |
| 7 | Cleanup & Summary | Resource release and training artifact verification |


---
## 1️⃣ Environment Setup & Project Bootstrap

> **🎯 Purpose:** Ensure project root directory is added to Python `sys.path` so that project modules (`config.py`, `utils.py`) can be imported seamlessly from subfolders.

| Item | Description |
|:---|:---|
| **Input** | Current working directory |
| **Output** | `sys.path` updated with project root path |


In [ ]:
# ============================================================
# PROJECT BOOTSTRAP — Ensure project root is on sys.path
# ============================================================
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'📁 Project Root: {PROJECT_ROOT}')


---
## 2️⃣ System Setup, Hardware Acceleration & Output Config

> **🎯 Purpose:** Import scientific packages, configure matplotlib aesthetics, and run **Automatic Hardware Detection** to query NVIDIA CUDA GPU and CPU multi-threading capabilities.

### ⚙️ How Automatic Hardware Detection Works
1. **XGBoost Test:** Tries to initialize `xgb.XGBRegressor(tree_method='hist', device='cuda')`. If CUDA fails, falls back to CPU multi-threading.
2. **CatBoost Test:** Tries to initialize `cb.CatBoostRegressor(task_type='GPU')`. If CUDA fails, falls back to CPU.
3. **LightGBM Strategy:** Configures `n_jobs=-1` to utilize all 12 CPU cores via OpenMP multi-threading (optimal for tabular data).

| Configuration | Path / Setting |
|:---|:---|
| **Output Reports & Plots** | `output/06_model_training/` |
| **Model Artifacts Registry** | `03_Models/advanced_models/` |


In [ ]:
import os, sys, time, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error, r2_score
import config, utils

import lightgbm as lgb
import catboost as cb
import xgboost as xgb

# ── Style Setup ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
COLORS = ['#f093fb', '#f5576c', '#4facfe', '#00c6ff', '#43e97b']

# Output directories
OUTPUT_DIR = Path(config.PROJECT_ROOT) / 'output' / '06_model_training'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path(getattr(config, 'MODELS_DIR', Path(config.PROJECT_ROOT) / '03_Models')) / 'advanced_models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 70)
print('💻 PRODUCTION HARDWARE & ACCELERATION PROFILE')
print('=' * 70)
print('   ⚡ XGBoost Regressor  : GPU Accelerated (device="cuda", tree_method="hist")')
print('   🐱 CatBoost Regressor : GPU Accelerated (task_type="GPU", float32 precision)')
print('   ⚡ LightGBM Regressor : CPU Multi-Threading (num_threads=12, OpenMP Windows)')
print(f'   📁 Output Directory   : {OUTPUT_DIR}')
print(f'   🤖 Models Registry   : {MODELS_DIR}')
print('=' * 70)


### 📊 2.1 Dataset Load & Out-of-Time Chronological Split

> **🎯 Purpose:** Load observations from `feature_store.parquet` and split into training and validation sets strictly by date to preserve temporal order.

### ⚙️ Why Target Log Transformation $\log(1 + y)$?
- Demand sales exhibit heavy right skewness ($0$ to $500+$ units).
- Fitting models on $\log(1 + y)$ stabilizes variance, prevents large-sale outlier dominance, and directly optimizes the **RMSLE** objective metric.
- During prediction, forecasts are mapped back using $\exp(y_{pred}) - 1$.

| Dataset Split | Date Range | Size |
|:---|:---|:---|
| **Train Set** | Past history up to `2017-07-31` | ~1.95M rows |
| **Validation Set** | Holdout period: `2017-08-01` → `2017-08-15` (16 days) | ~50K rows |


In [ ]:
# ── Load Feature Store & Perform Chronological Split ──
t0 = time.time()
FEATURE_STORE_PATH = Path(config.PROJECT_ROOT) / '01_Dataset' / 'features' / 'feature_store.parquet'

# Load 2,000,000 recent observations for fast memory-safe training
df = utils.load_feature_store_partial(rows=2_000_000, newest=True, verbose=True)

TARGET_COL = 'unit_sales'
IGNORED_COLS = {'date', 'unit_sales', 'is_return'}
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in IGNORED_COLS]

# Out-of-time train/val split (Val: last 16 days)
val_cutoff = pd.Timestamp('2017-08-01')
train_mask = df['date'] < val_cutoff
val_mask   = df['date'] >= val_cutoff

# Cast feature matrices to float32 for GPU CUDA compatibility & memory reduction
X_train = df.loc[train_mask, feature_cols].astype(np.float32)
y_train = df.loc[train_mask, TARGET_COL].values.astype(np.float32)
X_val   = df.loc[val_mask, feature_cols].astype(np.float32)
y_val   = df.loc[val_mask, TARGET_COL].values.astype(np.float32)

# Log-transformation: log(1 + y)
y_train_log = np.log1p(np.clip(y_train, 0, None))
y_val_log   = np.log1p(np.clip(y_val, 0, None))

print(f'\n✅ Data Preparation Complete in {time.time()-t0:.2f}s:')
print(f'   X_train : {X_train.shape[0]:,} rows × {X_train.shape[1]} features (float32)')
print(f'   X_val   : {X_val.shape[0]:,} rows × {X_val.shape[1]} features (float32)')
print(f'   Target  : log1p(unit_sales) transformation applied')
utils.memory_checkpoint('After Data Prep')


### 📊 2.2 Standardized Metric Evaluation Engine

> **🎯 Purpose:** Standardize metric calculation across all models (RMSLE, RMSE, MAE, MAPE, $R^2$) and track training/prediction runtimes.

| Metric | Formula | Interpretation |
|:---|:---|:---|
| **RMSLE** (Primary) | $\sqrt{\frac{1}{N}\sum (\log(1+y) - \log(1+\hat{y}))^2}$ | Penalizes relative percentage error |
| **RMSE** | $\sqrt{\frac{1}{N}\sum (y - \hat{y})^2}$ | Penalizes large absolute errors |
| **MAE** | $\frac{1}{N}\sum \|y - \hat{y}\|$ | Average absolute error units |
| **MAPE** | $\frac{100\%}{N}\sum \|\frac{y - \hat{y}}{y}\|$ | Percentage error on non-zero sales |


In [ ]:
# Standardized Evaluation Engine
advanced_results = []

def evaluate_advanced_model(model_name, y_val_true, y_pred_raw, train_time, pred_time):
    y_pred = np.expm1(np.clip(y_pred_raw, 0, None))
    y_true = np.clip(y_val_true, 0, None)
    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)
    mask  = y_true > 0
    mape  = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else 0.0
    metrics = {
        'Model': model_name,
        'RMSLE': float(rmsle),
        'RMSE': float(rmse),
        'MAE': float(mae),
        'MAPE': float(mape),
        'R2_Score': float(r2),
        'Train_Time_s': float(train_time),
        'Pred_Time_s': float(pred_time),
    }
    advanced_results.append(metrics)
    print(f'  📊 {model_name:20s} | RMSLE: {rmsle:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f} | R²: {r2:.4f} | Time: {train_time:.1f}s')
    return metrics


---
## 3️⃣ Model 1 — LightGBM Regressor (Leaf-Wise Tree Growth)

> **🎯 Purpose:** Train LightGBM regressor using leaf-wise tree splitting (`num_leaves=63`) and early stopping (50 rounds) on validation loss.

### ⚙️ Model Hyperparameters & Export Artifacts
- **Hyperparameters:** `n_estimators=2000`, `learning_rate=0.03`, `num_leaves=63`, `subsample=0.8`, `colsample_bytree=0.8`.
- **Feature Importance Plot:** Saved to `output/06_model_training/01_lightgbm_feature_importance.png` and displayed inline.
- **Model Artifact:** Saved to `03_Models/advanced_models/lightgbm_model.joblib`.


In [ ]:
print('⚡ Training LightGBM Regressor (CPU num_threads=12)...')
t0 = time.time()
model_lgb = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    num_threads=12,
    verbose=-1
)
model_lgb.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
train_time = time.time() - t0

t0 = time.time()
y_pred_log_lgb = model_lgb.predict(X_val)
pred_time = time.time() - t0

evaluate_advanced_model('LightGBM', y_val, y_pred_log_lgb, train_time, pred_time)

importances = model_lgb.feature_importances_
fi_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = plt.cm.plasma(np.linspace(0.2, 0.8, len(fi_df)))
ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors_bar, edgecolor='black', alpha=0.85)
ax.set_title('⚡ LightGBM — Top 15 Feature Importances (Gain)', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / '01_lightgbm_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

save_path_lgb = MODELS_DIR / 'lightgbm_model.joblib'
joblib.dump(model_lgb, save_path_lgb, compress=3)
print(f'💾 Model saved → {save_path_lgb.relative_to(config.PROJECT_ROOT)}')


---
## 4️⃣ Model 2 — CatBoost Regressor (Ordered Boosting & GPU Support)

> **🎯 Purpose:** Train CatBoost regressor using ordered boosting and native NVIDIA CUDA GPU acceleration (`task_type='GPU'`).

### ⚙️ Model Hyperparameters & Export Artifacts
- **Hyperparameters:** `iterations=1500`, `depth=7`, `learning_rate=0.04`, `eval_metric='RMSE'`.
- **Feature Importance Plot:** Saved to `output/06_model_training/02_catboost_feature_importance.png` and displayed inline.
- **Model Artifact:** Saved to `03_Models/advanced_models/catboost_model.joblib`.


In [ ]:
print('🐱 Training CatBoost Regressor (task_type="GPU")...')
t0 = time.time()
model_cb = cb.CatBoostRegressor(
    iterations=1500,
    learning_rate=0.04,
    depth=7,
    eval_metric='RMSE',
    task_type='GPU',
    random_seed=42,
    verbose=0
)
model_cb.fit(
    X_train, y_train_log,
    eval_set=(X_val, y_val_log),
    early_stopping_rounds=50
)
train_time = time.time() - t0

t0 = time.time()
y_pred_log_cb = model_cb.predict(X_val)
pred_time = time.time() - t0

evaluate_advanced_model('CatBoost', y_val, y_pred_log_cb, train_time, pred_time)

importances_cb = model_cb.get_feature_importance()
fi_cb_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances_cb}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, len(fi_cb_df)))
ax.barh(fi_cb_df['Feature'], fi_cb_df['Importance'], color=colors_bar, edgecolor='black', alpha=0.85)
ax.set_title('🐱 CatBoost — Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Feature Importance (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / '02_catboost_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

save_path_cb = MODELS_DIR / 'catboost_model.joblib'
joblib.dump(model_cb, save_path_cb, compress=3)
print(f'💾 Model saved → {save_path_cb.relative_to(config.PROJECT_ROOT)}')


---
## 5️⃣ Model 3 — XGBoost Regressor (Histogram CUDA GBDT)

> **🎯 Purpose:** Train XGBoost regressor using histogram-based binning (`tree_method='hist'`) and CUDA GPU acceleration (`device='cuda'`).

### ⚙️ Model Hyperparameters & Export Artifacts
- **Hyperparameters:** `n_estimators=1500`, `max_depth=7`, `learning_rate=0.04`, `subsample=0.8`, `colsample_bytree=0.8`.
- **Feature Importance Plot:** Saved to `output/06_model_training/03_xgboost_feature_importance.png` and displayed inline.
- **Model Artifact:** Saved to `03_Models/advanced_models/xgboost_model.joblib`.


In [ ]:
print('🚀 Training XGBoost Regressor (device="cuda", tree_method="hist")...')
t0 = time.time()
model_xgb = xgb.XGBRegressor(
    n_estimators=1500,
    learning_rate=0.04,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    random_state=42,
    early_stopping_rounds=50
)
model_xgb.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    verbose=False
)
train_time = time.time() - t0

t0 = time.time()
y_pred_log_xgb = model_xgb.predict(X_val)
pred_time = time.time() - t0

evaluate_advanced_model('XGBoost', y_val, y_pred_log_xgb, train_time, pred_time)

importances_xgb = model_xgb.feature_importances_
fi_xgb_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances_xgb}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = plt.cm.magma(np.linspace(0.2, 0.8, len(fi_xgb_df)))
ax.barh(fi_xgb_df['Feature'], fi_xgb_df['Importance'], color=colors_bar, edgecolor='black', alpha=0.85)
ax.set_title('🚀 XGBoost — Top 15 Feature Importances (Weight)', fontweight='bold')
ax.set_xlabel('Weight Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / '03_xgboost_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

save_path_xgb = MODELS_DIR / 'xgboost_model.joblib'
joblib.dump(model_xgb, save_path_xgb, compress=3)
print(f'💾 Model saved → {save_path_xgb.relative_to(config.PROJECT_ROOT)}')


---
## 6️⃣ Advanced Models Leaderboard & Comparative Diagnostics

> **🎯 Purpose:** Compile training metrics into a styled leaderboard table, compare GBDTs against baseline Random Forest (`RMSLE = 0.2873`), and export JSON/CSV benchmark reports.

| Export File | Purpose |
|:---|:---|
| `output/06_model_training/advanced_models_results.csv` | Summary table with all metrics |
| `output/06_model_training/training_metrics.json` | Machine-readable metrics dictionary |
| `output/06_model_training/04_advanced_models_comparison.png` | Grouped bar chart comparing RMSLE and Training Runtimes |


In [ ]:
df_adv_results = pd.DataFrame(advanced_results).sort_values('RMSLE').reset_index(drop=True)
df_adv_results['Rank'] = np.arange(1, len(df_adv_results) + 1)

print('=' * 80)
print('🏆 ADVANCED GBDT MODELS BENCHMARK LEADERBOARD')
print('=' * 80)
display(df_adv_results.style.format({
    'RMSLE': '{:.4f}', 'RMSE': '{:.2f}', 'MAE': '{:.2f}',
    'MAPE': '{:.2f}%', 'R2_Score': '{:.4f}', 'Train_Time_s': '{:.1f}s', 'Pred_Time_s': '{:.2f}s'
}).set_properties(**{'text-align': 'center'}))

df_adv_results.to_csv(OUTPUT_DIR / 'advanced_models_results.csv', index=False)
with open(OUTPUT_DIR / 'training_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(advanced_results, f, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors_cmp = COLORS[:len(df_adv_results)]
axes[0].bar(df_adv_results['Model'], df_adv_results['RMSLE'], color=colors_cmp, edgecolor='black', alpha=0.85)
axes[0].set_title('🏆 Model Comparison — Validation RMSLE (Lower = Better)', fontweight='bold')
axes[0].set_ylabel('RMSLE')
for i, v in enumerate(df_adv_results['RMSLE']):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

axes[1].bar(df_adv_results['Model'], df_adv_results['Train_Time_s'], color=colors_cmp, edgecolor='black', alpha=0.85)
axes[1].set_title('⏱️ Model Comparison — Training Runtime (Seconds)', fontweight='bold')
axes[1].set_ylabel('Seconds')
for i, v in enumerate(df_adv_results['Train_Time_s']):
    axes[1].text(i, v + 0.5, f'{v:.1f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / '04_advanced_models_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Benchmark report saved → {OUTPUT_DIR}')


---

<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 15px; border-radius: 10px; color: white; text-align: center; margin: 10px 0;'>

**End of Notebook 06 — Advanced Model Training Suite** ✅

| Stage | Next Notebook |
|:---|:---|
| ✅ Model Training | → [07_model_evaluation.ipynb](./07_model_evaluation.ipynb) |

</div>
